## 09.07 序列到序列学习（seq2seq） 参考答案


### 环境配置


In [1]:
import os
import sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "0"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    from torch import nn
    from torch.nn import functional as F
    import torch_npu
    import logging

warnings.filterwarnings("ignore", message="Permission mismatch")
warnings.filterwarnings("ignore", message="TASK_QUEUE_ENABLE")
warnings.filterwarnings("ignore", message="Cannot create tensor")
logging.getLogger('torch_npu').setLevel(logging.WARNING)
logging.getLogger('matplotlib').setLevel(logging.WARNING)
import matplotlib.pyplot as plt

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"
pypto.pypto_impl.DeviceInit()

### 练习 9.7.1

**题目：** 试着通过调整超参数来改善翻译效果。

**解答：** 将隐藏层和 Embedding 层维度修改为 256，batch_size 改为 256，学习率降低至 0.001，通常可以获得更好的翻译效果。

以下使用 `torch` 编程：


In [ ]:
import torch
from torch import nn
from d2l import torch as d2l

class Seq2SeqDecoder(d2l.Decoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + num_hiddens, num_hiddens, num_layers, dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)
    def init_state(self, enc_outputs, *args):
        return enc_outputs[1]
    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2)
        context = state[-1].repeat(X.shape[0], 1, 1)
        X_and_context = torch.cat((X, context), 2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    def forward(self, enc_X, dec_X, *args):
        enc_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_outputs, *args)
        return self.decoder(dec_X, dec_state)

embed_size, num_hiddens, num_layers, dropout = 256, 256, 2, 0.1
batch_size, num_steps = 256, 10
lr, num_epochs = 0.001, 300
train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
encoder = d2l.Seq2SeqEncoder(len(src_vocab), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqDecoder(len(tgt_vocab), embed_size, num_hiddens, num_layers, dropout)
net = EncoderDecoder(encoder, decoder)
d2l.train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, d2l.try_gpu())

使用 `PyPTO` 编程：

PyPTO 版采用与 torch 版相同的调参配置（embed/hidden=256、batch=256、lr=0.001、300 epochs），在 NPU 上完成训练。


In [2]:
from src.utils import load_data_nmt, Encoder, Decoder, EncoderDecoder
from src.utils import Timer, Accumulator, sequence_mask, grad_clipping
from src.pypto_ops import PyPTOLinear, PyPTOGRU

class Seq2SeqEncoderPyPTO(Encoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = PyPTOGRU(embed_size, num_hiddens, num_layers, dropout)  # 多层 GRU
    def forward(self, X, *args):
        X = self.embedding(X).permute(1, 0, 2).contiguous()
        output, state = self.rnn(X)
        return output, state

class Seq2SeqDecoderPyPTO(Decoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = PyPTOGRU(embed_size + num_hiddens, num_hiddens, num_layers, dropout)  # 多层 GRU
        self.dense = PyPTOLinear(num_hiddens, vocab_size)
    def init_state(self, enc_outputs, *args):
        return enc_outputs[1]
    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2).contiguous()
        context = state[-1].repeat(X.shape[0], 1, 1)
        X_and_context = torch.cat((X, context), 2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state

class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """带遮蔽的 softmax 交叉熵损失函数"""
    def __init__(self):
        super().__init__()
        self.reduction = 'none'
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)
        weights = sequence_mask(weights, valid_len)
        unweighted_loss = super().forward(pred.permute(0, 2, 1), label)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss

def train_seq2seq(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """训练序列到序列模型"""
    def xavier_init_weights(m):
        if type(m) in (nn.Linear, PyPTOLinear):
            nn.init.xavier_uniform_(m.weight)
        if type(m) == PyPTOGRU:
            for name, param in m.named_parameters():
                if "W_" in name:
                    nn.init.xavier_uniform_(param)
    net.apply(xavier_init_weights)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()
    net.train()
    for epoch in range(num_epochs):
        timer = Timer()
        metric = Accumulator(2)
        for batch in data_iter:
            optimizer.zero_grad()
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            bos = torch.tensor([tgt_vocab['<bos>']] * Y.shape[0],
                          device=device).reshape(-1, 1)
            dec_input = torch.cat([bos, Y[:, :-1]], 1)
            Y_hat, _ = net(X, dec_input, X_valid_len)
            l = loss(Y_hat, Y, Y_valid_len)
            l.sum().backward()
            grad_clipping(net, 1)
            num_tokens = Y_valid_len.sum()
            optimizer.step()
            with torch.no_grad():
                metric.add(l.sum(), num_tokens)
    print(f'loss {metric[0]/metric[1]:.3f}, {metric[1]/timer.stop():.1f} '
        f'tokens/sec on {str(device)}')

batch_size, num_steps = 256, 10
train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size, num_steps)
encoder = Seq2SeqEncoderPyPTO(len(src_vocab), 256, 256, 2, 0.1)
decoder = Seq2SeqDecoderPyPTO(len(tgt_vocab), 256, 256, 2, 0.1)
net = EncoderDecoder(encoder, decoder).to(device)

# JIT 编译预热
X_batch, X_len, Y_batch, Y_len = next(iter(train_iter))
X_batch = X_batch.to(device); Y_batch = Y_batch.to(device)
X_len = X_len.to(device); Y_len = Y_len.to(device)
bos = torch.tensor([tgt_vocab['<bos>']] * Y_batch.shape[0], device=device).reshape(-1, 1)
dec_input = torch.cat([bos, Y_batch[:, :-1]], 1)
loss = MaskedSoftmaxCELoss()
Y_hat, _ = net(X_batch, dec_input, X_len)
l = loss(Y_hat, Y_batch, Y_len)
l.sum().backward(); net.zero_grad()

# 训练
train_seq2seq(net, train_iter, 0.001, 300, tgt_vocab, device)


loss 0.017, 2919.9 tokens/sec on npu:0


### 练习 9.7.2

**题目：** 重新运行实验并在计算损失时不使用遮蔽，可以观察到什么结果？

**解答：** 注释掉 `sequence_mask()` 后，不再根据有效长度屏蔽无效位置，导致所有位置（包括填充位置）都参与损失计算，可能对模型训练产生负面影响。

以下使用 `torch` 编程：


In [ ]:
class NoMaskSoftmaxCELoss(nn.CrossEntropyLoss):
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)  # 不使用 sequence_mask
        self.reduction = 'none'
        unweighted_loss = super().forward(pred.permute(0, 2, 1), label)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss
loss = NoMaskSoftmaxCELoss()
print(loss(torch.ones(3, 4, 10), torch.ones((3, 4), dtype=torch.long), torch.tensor([4, 2, 0])))


使用 `PyPTO` 编程：


In [3]:
from src.utils import sequence_mask
from src.pypto_ops import PyPTOSoftmaxCrossEntropyLoss


class PyPTOSoftmaxCELossNoMask:
    """不使用遮蔽的 softmax 交叉熵损失（PyPTO 版）。

    与正文 PyPTOMaskedSoftmaxCELoss 的区别：不调用 sequence_mask，
    填充位置也参与损失计算。
    """

    def __call__(self, pred, label, valid_len):
        return self.forward(pred, label, valid_len)

    def forward(self, pred, label, valid_len):
        B, T, V = pred.shape
        per_token_loss = PyPTOSoftmaxCrossEntropyLoss.apply(
            pred.reshape(-1, V), label.reshape(-1), V
        ).reshape(B, T)
        return per_token_loss.mean(dim=1)


class PyPTOMaskedSoftmaxCELoss:
    """带遮蔽的 softmax 交叉熵损失（PyPTO 版，用于对照）。"""

    def __call__(self, pred, label, valid_len):
        return self.forward(pred, label, valid_len)

    def forward(self, pred, label, valid_len):
        B, T, V = pred.shape
        per_token_loss = PyPTOSoftmaxCrossEntropyLoss.apply(
            pred.reshape(-1, V), label.reshape(-1), V
        ).reshape(B, T)
        weights = torch.ones_like(per_token_loss)
        weights = sequence_mask(weights, valid_len)
        return (per_token_loss * weights).mean(dim=1)


# 与正文一致：三个相同序列，有效长度分别为 4、2、0
pred = torch.ones(3, 4, 10, device=device)
label = torch.ones((3, 4), dtype=torch.long, device=device)
valid_len = torch.tensor([4, 2, 0], device=device)

l_no_mask = PyPTOSoftmaxCELossNoMask()(pred, label, valid_len)
l_masked = PyPTOMaskedSoftmaxCELoss()(pred, label, valid_len)
print('不使用遮蔽:', l_no_mask.cpu().numpy())
print('使用遮蔽:  ', l_masked.cpu().numpy())


不使用遮蔽: [2.3025851 2.3025851 2.3025851]
使用遮蔽:   [2.3025851 1.1512926 0.       ]


### 练习 9.7.3

**题目：** 如果编码器和解码器的层数或隐藏单元数不同，如何初始化解码器的隐状态？

**解答：** 可通过以下方法：
1. 使用线性变换将编码器隐藏状态维度调整为解码器所需的维度
2. 使用零向量进行初始化。

注意：线性变换（全连接层）只调整隐藏单元数（最后一维）。若编码器/解码器层数不同，还需适配隐状态的层数维度：例如只取编码器顶层（最后一层）隐状态，再复制或补零扩展为解码器层数。下面代码演示隐藏单元数不同（16 → 10）的情形：

以下使用 `torch` 编程（全连接层变换）：


In [ ]:
class Seq2SeqDecoder(d2l.Decoder):
    def __init__(self, vocab_size, embed_size, num_hiddens_enc, num_hiddens_dec, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.GRU(embed_size + num_hiddens_dec, num_hiddens_dec, num_layers, dropout=dropout)
        self.fc = nn.Linear(num_hiddens_enc, num_hiddens_dec)
        self.dense = nn.Linear(num_hiddens_dec, vocab_size)
    def init_state(self, enc_outputs, *args):
        return self.fc(enc_outputs[1])
    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2)
        context = state[-1].repeat(X.shape[0], 1, 1)
        X_and_context = torch.cat((X, context), 2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state

encoder = d2l.Seq2SeqEncoder(vocab_size=10, embed_size=8, num_hiddens=16, num_layers=2)
encoder.eval()
X = torch.zeros((4, 7), dtype=torch.long)
decoder = Seq2SeqDecoder(10, 8, 16, 10, 2)
decoder.eval()
state = decoder.init_state(encoder(X))
output, state = decoder(X, state)
print('Output shape:', output.shape)


使用 `PyPTO` 编程：


In [4]:
from src.utils import Encoder, Decoder
from src.pypto_ops import PyPTOLinear, PyPTOGRU


class Seq2SeqEncoderPyPTO(Encoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = PyPTOGRU(embed_size, num_hiddens, num_layers, dropout)

    def forward(self, X, *args):
        X = self.embedding(X).permute(1, 0, 2).contiguous()
        output, state = self.rnn(X)
        return output, state


class Seq2SeqDecoderPyPTOProj(Decoder):
    """编码器/解码器隐藏单元数不同时的解码器（PyPTO 版）。

    通过 PyPTOLinear 全连接层将编码器隐状态投影为解码器所需维度。
    """

    def __init__(self, vocab_size, embed_size, num_hiddens_enc,
                 num_hiddens_dec, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = PyPTOGRU(embed_size + num_hiddens_dec,
                            num_hiddens_dec, num_layers, dropout)
        self.fc = PyPTOLinear(num_hiddens_enc, num_hiddens_dec)
        self.dense = PyPTOLinear(num_hiddens_dec, vocab_size)

    def init_state(self, enc_outputs, *args):
        return self.fc(enc_outputs[1])  # 线性变换到解码器维度

    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2).contiguous()
        context = state[-1].repeat(X.shape[0], 1, 1)
        X_and_context = torch.cat((X, context), 2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state


# 验证：编码器隐藏单元数 16，解码器 10，通过 fc 投影初始化解码器状态
encoder = Seq2SeqEncoderPyPTO(vocab_size=10, embed_size=8,
                              num_hiddens=16, num_layers=2).to(device)
encoder.eval()
X = torch.zeros((4, 7), dtype=torch.long, device=device)
enc_outputs = encoder(X)
decoder = Seq2SeqDecoderPyPTOProj(10, 8, 16, 10, 2).to(device)
decoder.eval()
state = decoder.init_state(enc_outputs)
print('投影后状态形状:', state.shape)  # (2, 4, 10)
output, state = decoder(X, state)
print('Output shape:', output.shape)


投影后状态形状: torch.Size([2, 4, 10])


Output shape: torch.Size([4, 7, 10])


### 练习 9.7.4

**题目：** 在训练中用前一时间步的预测输入代替强制教学，对性能有何影响？

**解答：** 使用自回归训练方式会导致：
1. 训练速度变慢（逐步预测而非一次性使用目标序列）
2. 累积误差（错误会逐步扩大）
3. 训练稳定性降低
4. 可能降低模型性能。但注意力机制等方法可有效改善。


### 练习 9.7.5

**题目：** 用 LSTM 替换 GRU 重新运行实验。

**解答：** 将编码器和解码器中的 `nn.GRU` 替换为 `nn.LSTM`。LSTM 有三门结构，保留了记忆元，可能更好地捕捉长期依赖。

以下使用 `torch` 编程：


In [ ]:
# 重新定义标准 5 参数版本的 Seq2SeqDecoder（LSTM 版）
class Seq2SeqDecoder(d2l.Decoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.LSTM(embed_size + num_hiddens, num_hiddens, num_layers, dropout=dropout)
        self.dense = nn.Linear(num_hiddens, vocab_size)
    def init_state(self, enc_outputs, *args):
        return enc_outputs[1]  # LSTM state 为 (h, c) 二元组
    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2)
        context = state[0][-1].repeat(X.shape[0], 1, 1)  # 取顶层隐状态 h
        X_and_context = torch.cat((X, context), 2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state

class Seq2SeqEncoderLSTM(d2l.Encoder):
    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.LSTM(embed_size, num_hiddens, num_layers, dropout=dropout)
    def forward(self, X, *args):
        X = self.embedding(X).permute(1, 0, 2)
        output, state = self.rnn(X)
        return output, state

class EncoderDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    def forward(self, enc_X, dec_X, *args):
        enc_outputs = self.encoder(enc_X, *args)
        dec_state = self.decoder.init_state(enc_outputs, *args)
        return self.decoder(dec_X, dec_state)

embed_size, num_hiddens, num_layers, dropout = 32, 32, 2, 0.1
batch_size, num_steps = 64, 10
lr, num_epochs = 0.005, 500
train_iter, src_vocab, tgt_vocab = d2l.load_data_nmt(batch_size, num_steps)
encoder = Seq2SeqEncoderLSTM(len(src_vocab), embed_size, num_hiddens, num_layers, dropout)
decoder = Seq2SeqDecoder(len(tgt_vocab), embed_size, num_hiddens, num_layers, dropout)
net = EncoderDecoder(encoder, decoder)
d2l.train_seq2seq(net, train_iter, lr, num_epochs, tgt_vocab, d2l.try_gpu())


使用 `PyPTO` 编程：

PyPTO 版采用与 torch 版一致的 2 层 LSTM 结构；为控制 NPU 训练时长，仅训练 10 epochs（torch 版为 500），仅作正确性演示。


In [5]:
from src.utils import Encoder, Decoder, EncoderDecoder, load_data_nmt
from src.utils import Timer, Accumulator, grad_clipping, sequence_mask
from src.pypto_ops import PyPTOLinear, PyPTOLSTM

class MaskedSoftmaxCELoss(nn.CrossEntropyLoss):
    """带遮蔽的 softmax 交叉熵损失（PyPTO 版，self-contained）。"""
    def __init__(self):
        super().__init__()
        self.reduction = 'none'
    def forward(self, pred, label, valid_len):
        weights = torch.ones_like(label)
        weights = sequence_mask(weights, valid_len)
        unweighted_loss = super().forward(pred.permute(0, 2, 1), label)
        weighted_loss = (unweighted_loss * weights).mean(dim=1)
        return weighted_loss


class Seq2SeqEncoderPyPTOLSTM(Encoder):
    """LSTM 编码器（PyPTO 版）：PyPTOLSTM 单向，支持多层。"""

    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers=1, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = PyPTOLSTM(embed_size, num_hiddens, num_layers, dropout)

    def forward(self, X, *args):
        X = self.embedding(X).permute(1, 0, 2).contiguous()
        output, state = self.rnn(X)
        return output, state


class Seq2SeqDecoderPyPTOLSTM(Decoder):
    """LSTM 解码器（PyPTO 版）：state 为 (h, c) 二元组。"""

    def __init__(self, vocab_size, embed_size, num_hiddens, num_layers=1, dropout=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = PyPTOLSTM(embed_size + num_hiddens, num_hiddens, num_layers, dropout)
        self.dense = PyPTOLinear(num_hiddens, vocab_size)

    def init_state(self, enc_outputs, *args):
        return enc_outputs[1]

    def forward(self, X, state):
        X = self.embedding(X).permute(1, 0, 2).contiguous()
        context = state[0][-1].repeat(X.shape[0], 1, 1)  # 取顶层隐状态 h
        X_and_context = torch.cat((X, context), 2)
        output, state = self.rnn(X_and_context, state)
        output = self.dense(output).permute(1, 0, 2)
        return output, state


def train_seq2seq_lstm(net, data_iter, lr, num_epochs, tgt_vocab, device):
    """训练序列到序列模型（PyPTO LSTM 版）"""

    def xavier_init_weights(m):
        if type(m) == PyPTOLinear:
            nn.init.xavier_uniform_(m.weight)
        if type(m) == PyPTOLSTM:
            for name, param in m.named_parameters():
                if "W_" in name:
                    nn.init.xavier_uniform_(param)

    net.apply(xavier_init_weights)
    net.to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    loss = MaskedSoftmaxCELoss()
    net.train()
    for epoch in range(num_epochs):
        timer = Timer()
        metric = Accumulator(2)
        for batch in data_iter:
            optimizer.zero_grad()
            X, X_valid_len, Y, Y_valid_len = [x.to(device) for x in batch]
            bos = torch.tensor([tgt_vocab['<bos>']] * Y.shape[0],
                               device=device).reshape(-1, 1)
            dec_input = torch.cat([bos, Y[:, :-1]], 1)
            Y_hat, _ = net(X, dec_input, X_valid_len)
            l = loss(Y_hat, Y, Y_valid_len)
            l.sum().backward()
            grad_clipping(net, 1)
            num_tokens = Y_valid_len.sum()
            optimizer.step()
            with torch.no_grad():
                metric.add(l.sum(), num_tokens)
    print(f'loss {metric[0] / metric[1]:.3f}, {metric[1] / timer.stop():.1f} '
          f'tokens/sec on {str(device)}')


batch_size, num_steps = 64, 10
train_iter, src_vocab, tgt_vocab = load_data_nmt(batch_size, num_steps)
encoder = Seq2SeqEncoderPyPTOLSTM(len(src_vocab), 32, 32, 2, 0.1)
decoder = Seq2SeqDecoderPyPTOLSTM(len(tgt_vocab), 32, 32, 2, 0.1)
net = EncoderDecoder(encoder, decoder).to(device)

# JIT 编译预热
X_batch, X_len, Y_batch, Y_len = next(iter(train_iter))
X_batch = X_batch.to(device); Y_batch = Y_batch.to(device)
X_len = X_len.to(device); Y_len = Y_len.to(device)
bos = torch.tensor([tgt_vocab['<bos>']] * Y_batch.shape[0], device=device).reshape(-1, 1)
dec_input = torch.cat([bos, Y_batch[:, :-1]], 1)
loss = MaskedSoftmaxCELoss()
Y_hat, _ = net(X_batch, dec_input, X_len)
l = loss(Y_hat, Y_batch, Y_len)
l.sum().backward(); net.zero_grad()

# 训练
train_seq2seq_lstm(net, train_iter, 0.005, 10, tgt_vocab, device)

loss 0.242, 877.1 tokens/sec on npu:0


### 练习 9.7.6

**题目：** 有没有其他方法来设计解码器的输出层？

**解答：** 可以在解码器的输出层应用注意力机制，通过计算注意力权重对编码器隐藏状态进行加权求和，使解码器将注意力集中在输入序列的相关部分，有效捕捉重要信息，提高生成准确性。


---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
